# Mapping migration

## Set up

To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [1]:
%store -r

Unable to restore variable 'gbif_ecoregion_gdf', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>


You will also need to import any libraries you are using in this notebook:

In [ ]:
### import libraries

import os
import pathlib
import pandas as pd
import geopandas as gpd
import earthpy as et
from pathlib import Path
import sys
from importlib.metadata import version
import requests

print("Python:", sys.executable)
print("EarthPy:", version("earthpy"))
print("Setuptools:", version("setuptools"))

Python: c:\Users\gekek\miniconda3\envs\earth-analytics-python\python.exe
EarthPy: 0.10.0
Setuptools: 80.9.0


## Step 4: Connect the GBIF observations to the ecoregions

### Step 4A: Identify the ecoregion for each observation

You can combine the ecoregions and the observations **spatially** using a method called `.sjoin()`, which stands for spatial join.

Check out the [`geopandas` documentation on spatial joins](https://geopandas.org/en/stable/docs/user_guide/mergingdata.html#spatial-joins) to learn more about spatial joins and how they work.

To perform your spatial join, you'll need to identify the correct values for the `how =` and `predicate =` parameters of the spatial join.

In [3]:
### attach ecoregion attributes to each observation
gbif_ecoregion_gdf = gbif_gdf.sjoin(
    ecoregions_gdf.to_crs(gbif_gdf.crs),
    how="inner",
    predicate="within",
)

### check out the joined data
gbif_ecoregion_gdf.head()

,gbifID,datasetKey,occurrenceID,kingdom,phylum,class,order,family,genus,species,...,ECO_BIOME_,NNH,ECO_ID,SHAPE_LENG,SHAPE_AREA,NNH_NAME,COLOR,COLOR_BIO,COLOR_NNH,LICENSE
1,4923515059,36f15a36-6b45-442e-9c31-cd633423aee0,80de1f4e-30b8-4ffc-88e7-654a025164a5,Animalia,Chordata,Aves,Passeriformes,Turdidae,Catharus,Catharus fuscescens,...,NE08,4,387,62.955857,24.173692,Nature Imperiled,#A3FC62,#FEFF73,#EE1E23,CC-BY 4.0
2,4923522410,36f15a36-6b45-442e-9c31-cd633423aee0,da8061c1-aecd-49f8-84a7-31c39358f2cf,Animalia,Chordata,Aves,Passeriformes,Turdidae,Catharus,Catharus fuscescens,...,NE08,4,387,62.955857,24.173692,Nature Imperiled,#A3FC62,#FEFF73,#EE1E23,CC-BY 4.0
3,4923520798,36f15a36-6b45-442e-9c31-cd633423aee0,bf61d7c8-b847-4250-8d5a-798694a2ddfc,Animalia,Chordata,Aves,Passeriformes,Turdidae,Catharus,Catharus fuscescens,...,NE08,4,387,62.955857,24.173692,Nature Imperiled,#A3FC62,#FEFF73,#EE1E23,CC-BY 4.0
4,4923520314,36f15a36-6b45-442e-9c31-cd633423aee0,665d4e55-daaa-40d5-8339-e4b31bb303f4,Animalia,Chordata,Aves,Passeriformes,Turdidae,Catharus,Catharus fuscescens,...,NE08,4,387,62.955857,24.173692,Nature Imperiled,#A3FC62,#FEFF73,#EE1E23,CC-BY 4.0
5,4923522547,36f15a36-6b45-442e-9c31-cd633423aee0,f25bf3cc-a766-416f-8389-79c990d9496f,Animalia,Chordata,Aves,Passeriformes,Turdidae,Catharus,Catharus fuscescens,...,NE08,4,387,62.955857,24.173692,Nature Imperiled,#A3FC62,#FEFF73,#EE1E23,CC-BY 4.0


### Step 4B: Count the observations in each ecoregion each month

First, group the GBIF observations by ecoregion:
1. Replace `columns_to_group_by` with a list of columns. Keep in mind that you will end up with one row for each group – you want to count the observations in each ecoregion by month.
2. Select only month/ecoregion combinations that have more than one occurrence recorded, since a single occurrence could be an error.
3. Use the `.groupby()` and `.mean()` methods to compute the mean occurrences by ecoregion and by month.
4. Run the code – it will normalize the number of occurrences by month and ecoregion.

In [4]:
### summarize the occurrences by ecoregion/month combination
occurrence_df = (
    gbif_ecoregion_gdf

    ### group by each ecoregion and month
    .groupby(["ECO_ID", "month"])

    ### count observations
    .agg(occurrences=("gbifID", "count"))
)

### retain combinations with more than one observation
occurrence_df = occurrence_df[
    occurrence_df["occurrences"] > 1
].copy()

### calculate the mean # observations by ecoregion
mean_occurrences_by_ecoregion = (
    occurrence_df
    .groupby(level="ECO_ID")["occurrences"]
    .mean()
)

### calculate the mean # observations by month
mean_occurrences_by_month = (
    occurrence_df
    .groupby(level="month")["occurrences"]
    .mean()
)

### check the counts
occurrence_df.head()

occurrences
ECO_ID month             
328    4                2
       5             3023
       6             3242
       7             1823
       8              465

### Step 4C: Normalize the observations

Divide occurrences by the mean occurrences by month and by ecoregion.

In [5]:
### normalize by space and time for sampling effort
occurrence_df["norm_occurrences"] = (
    occurrence_df["occurrences"]
    .div(mean_occurrences_by_ecoregion, level="ECO_ID")
    .div(mean_occurrences_by_month, level="month")
)

### check it out
occurrence_df

occurrences  norm_occurrences
ECO_ID month                               
328    4                2          0.000010
       5             3023          0.001701
       6             3242          0.002054
       7             1823          0.002214
       8              465          0.001914
...                   ...               ...
613    4               13          0.009370
       9                5          0.001905
       10              11          0.033792
672    9               28          0.005575
       10               9          0.014447

[309 rows x 2 columns]

## Wrap up Step 4

Don’t forget to store your variables so you can use them in other notebooks! Replace `var1` and `var2` with the variable you want to save, separated by spaces.

In [6]:
%store occurrence_df gbif_ecoregion_gdf

Stored 'occurrence_df' (DataFrame)
Stored 'gbif_ecoregion_gdf' (GeoDataFrame)


Finally, be sure to `Restart` and `Run all` to make sure your notebook
works all the way through!